# EA3 - Observabilidad con Grafana (servicios locales vs nube)

## Objetivos
- Ver los servicios del entorno **funcionando en tiempo real** en Grafana.
- Entender la observabilidad como otro servicio que se opera en local, con su
  equivalente **gestionado** en la nube.
- Recorrer el pipeline `Kafka → Spark Streaming → Postgres → Grafana`.

> **Requiere el perfil `completo`:** `docker compose --profile completo up -d`

## El stack de observabilidad

| Pieza local | Rol | GCP | AWS | Azure |
|---|---|---|---|---|
| **Grafana** | Dashboards | Cloud Monitoring / Looker Studio | CloudWatch / QuickSight | Azure Monitor / Power BI |
| **Prometheus** | Métricas (TSDB) | Managed Prometheus | Amazon Managed Prometheus | Azure Monitor |
| **cAdvisor** | CPU/RAM por contenedor | métricas de GKE | ECS/EKS | AKS |
| **kafka-exporter** | Lag/throughput de Kafka | métricas de Pub/Sub | Kinesis / MSK | Event Hubs |
| **Postgres** | Serving layer (datos) | BigQuery | Redshift | Synapse |
| **Spark Streaming** | Procesamiento | Dataflow | Kinesis Analytics | Stream Analytics |

**Idea fuerza:** en local *vos* operás Prometheus, los exporters y Grafana. En la
nube todo eso es gestionado: pagás por uso y el proveedor lo mantiene. Los
conceptos (métricas, lag, dashboards) son idénticos.

## 1. Abrí Grafana

En el navegador: **http://localhost:3000** (o el puerto que pusiste en
`GRAFANA_PORT` dentro de `.env`). Entrás directo (acceso anónimo de lectura).

En la carpeta **Big Data** vas a ver dos dashboards:

| Dashboard | Secciones | Tipos de panel que verás |
|---|---|---|
| **🩺 Infraestructura** | Salud de servicios, CPU/RAM, Kafka | Stat (UP/DOWN), Time series, umbrales de lag |
| **📊 Negocio en vivo** | KPIs, series temporales, comparación, tabla | Stat, Time series, Bar gauge, Pie chart, Table |

Cada sección del dashboard indica **qué tipo de panel es** y para qué sirve.
El de negocio estará vacío hasta que generes datos (pasos 2 y 3, o el script
`scripts/iniciar_dashboard_vivo.sh` desde tu terminal).

## 2. Generá transacciones hacia Kafka

**Opción rápida (desde tu terminal, fuera de Jupyter):**
```bash
./scripts/iniciar_dashboard_vivo.sh        # Linux/Mac
scripts\iniciar_dashboard_vivo.bat         # Windows
```

**Opción manual (desde este notebook):** la siguiente celda lanza el generador
en segundo plano (120 s de eventos a 8 tx/s).

In [ ]:
import subprocess, sys

# Produce transacciones al topic 'transacciones' durante 120s, en segundo plano.
proc = subprocess.Popen([
    sys.executable, "/home/jovyan/scripts/generar_datos_streaming.py",
    "--tipo", "transacciones", "--velocidad", "8",
    "--duracion", "120", "--topic", "transacciones",
])
print("Generador lanzado (PID", proc.pid, ") — produciendo a 'transacciones' por 120s.")
print("Abrí Grafana → Big Data → 📊 Negocio en vivo y mirá el panel 'Throughput'.")

## 3. Arrancá el job de streaming Kafka → Postgres

Este job consume el topic, agrega por región y escribe en Postgres (lo que lee
Grafana). **Queda corriendo**: cuando quieras frenarlo, interrumpí el kernel
(botón ■ / Kernel → Interrupt).

> Mientras corre, mirá el dashboard **📊 Negocio en vivo** en Grafana: se va
> actualizando solo cada pocos segundos.

In [ ]:
# Ejecuta el job de streaming (bloqueante; interrumpí el kernel para frenar).
%run /home/jovyan/scripts/streaming_a_postgres.py

## 4. Recorré los dashboards (guía de paneles)

Con el generador y el job corriendo, abrí Grafana y recorré cada sección:

### 📊 Negocio en vivo
| Panel | Tipo | Qué deberías ver |
|---|---|---|
| Transacciones totales | Stat + sparkline | Número subiendo |
| Throughput tx/min | Time series (barras) | ~480 tx/min con generador a 8/s |
| Monto por región | Time series (líneas) | 3 líneas: Norte, Centro, Sur |
| Bar gauge / Pie chart | Comparación | Centro suele liderar (más tiendas) |
| Últimos lotes | Table | Filas nuevas cada ~5 s |

### 🩺 Infraestructura
| Panel | Tipo | Qué deberías ver |
|---|---|---|
| Prometheus / cAdvisor / Kafka | Stat UP/DOWN | Verde = UP |
| CPU por contenedor | Time series | Pico en `bigdata-jupyter` (Spark) |
| Velocidad de ingesta | Time series | ~8 msg/s en `transacciones` |
| Lag de consumidores | Time series + umbrales | Verde si Spark consume al día |

> **Pregunta de reflexión:** ¿qué panel usarías para alertar si el pipeline se cae?
> (Pista: lag de Kafka o último lote = 0)

---
## Ejercicio 1 — Panel "Top 5 productos por monto"

En el dashboard de **Negocio**, los datos se agregan por *región*, no por
*producto*. Para tener "top productos":

1. Modificá `scripts/streaming_a_postgres.py` para agrupar también por
   `producto` (o creá una tabla nueva `ventas_producto`).
2. En Grafana → dashboard Negocio → **Add panel** → tipo *Bar chart* → datasource
   **Postgres-Analytics** → query:
   ```sql
   SELECT producto, SUM(monto_total) AS monto
   FROM ventas_producto
   GROUP BY producto ORDER BY monto DESC LIMIT 5;
   ```
3. Guardá el panel y observá cómo se llena al correr el job.

**Símil cloud:** esto mismo lo harías en Looker Studio / QuickSight sobre
BigQuery / Redshift.

## Ejercicio 2 — Panel de velocidad de ingesta Kafka (infraestructura)

En el dashboard de **Infraestructura** ya hay un panel *Velocidad de ingesta (msg/s)*.
Tu tarea:

1. Abrí el dashboard **🩺 Infraestructura** y localizá ese panel.
2. Con el generador activo, confirmá que ves ~8 msg/s en el topic `transacciones`.
3. **Add panel** → tipo *Stat* → datasource **Prometheus** → query:
   ```promql
   sum(rate(kafka_topic_partition_current_offset{topic="transacciones"}[1m]))
   ```
4. Poné un umbral: verde si > 1, rojo si = 0 (sin datos entrando).
5. Guardá el panel y explicá en una frase qué mide.

**Símil cloud:** es la misma métrica de throughput que verías en CloudWatch (MSK)
o Cloud Monitoring (Pub/Sub).

---
## Cierre — ¿quién opera qué?

| | Local (este entorno) | Nube |
|---|---|---|
| Instalar/actualizar Prometheus, Grafana | **Vos** | El proveedor |
| Escalar el almacenamiento de métricas | **Vos** | Automático |
| Construir dashboards | Igual (Grafana/Looker/QuickSight) | Igual |
| Conceptos (métricas, lag, paneles) | **Idénticos** | **Idénticos** |

La observabilidad es transversal: una vez que entendés qué medir (CPU, memoria,
lag, throughput) y cómo visualizarlo, la herramienta concreta —Grafana o el
servicio gestionado de la nube— es secundaria.